# Above-ground biomass estimation from Sentinel-1 and Sentinel-2

ENV-540 Project 2. This notebook runs the experiments; the helper functions live in `src/`.

1. Setup
2. Preprocessing (raw tiles to `.npz` + normalization stats)
3. Data exploration
4. Training: S1 only, S2 only, S1+S2 (early and mid fusion)
5. Test-set evaluation
6. Contribution analysis (permutation, mean replacement, retraining ablation)
7. Discussion

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, json
from pathlib import Path
sys.path.insert(0, str(Path("src").resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from bands import S1_BANDS, S2_BANDS, GROUPS
from prepare_data import prepare
from dataset import split_files, load_stats
from train import make_cfg, train_model, get_device
from evaluate import evaluate_run, plot_predictions
from importance import compute_importance, plot_importance

DATA_RAW = Path("data/raw")         # downloaded dataset
DATA = Path("data/processed")       # output of preprocessing
RUNS = Path("runs")

# DEBUG=True: tiny models and few epochs to check the pipeline end to end
DEBUG = True
TRAIN_KW = dict(epochs=2, max_train=32, base=16, bs=4) if DEBUG else dict(epochs=50, base=32, bs=8)

print("device:", get_device())

## 2. Preprocessing

The data come already split into `AGB_training_set` (240 tiles) and `AGB_test_set` (60 tiles), with one acquisition per
tile (S1 and S2 from month 10). `prepare()` converts each tile to a `.npz`, marks missing values as NaN
(S1 = -9999, S2 all-zero pixels, CLP = 255), writes a per-tile quality table and computes normalization stats.
24 training tiles are held out for validation (random, seeded).
Already processed tiles are skipped, so re-running this cell is cheap.

In [ ]:
stats = prepare(DATA_RAW, DATA)
tr_files, va_files, te_files = split_files(DATA)
print(f"train={len(tr_files)} val={len(va_files)} test={len(te_files)}")

In [ ]:
# Data-quality summary: missing S1 orbits, empty S2 tiles, cloudiness, AGB outliers
qc = {s: pd.read_csv(DATA / s / "tiles.csv") for s in ("train", "test")}
for s, q in qc.items():
    print(f"{s}: ASC missing {(q.asc_missing > 0.5).sum()}, DSC missing {(q.dsc_missing > 0.5).sum()}, "
          f"S2 >50% missing {(q.s2_missing > 0.5).sum()}, mean CLP > 20: {(q.clp_mean > 20).sum()}, "
          f"AGB max > 500: {(q.agb_max > 500).sum()} tiles")
qc["train"].sort_values("s2_missing", ascending=False).head(8)

## 3. Data exploration

In [ ]:
def stretch(img, p=(2, 98)):
    lo, hi = np.nanpercentile(img, p)
    return np.clip((img - lo) / (hi - lo + 1e-9), 0, 1)

d = np.load(tr_files[0])
s1, s2, agb = d["s1"], d["s2"], d["agb"]
b = {n: s2[i] for i, n in enumerate(S2_BANDS)}

fig, ax = plt.subplots(2, 3, figsize=(14, 9))
panels = [
    (np.dstack([stretch(b["B4"]), stretch(b["B3"]), stretch(b["B2"])]), "S2 RGB (B4,B3,B2)", None),
    (np.dstack([stretch(b["B8"]), stretch(b["B4"]), stretch(b["B3"])]), "S2 false colour (B8,B4,B3)", None),
    (b["CLP"], "CLP (cloud probability)", "gray"),
    (s1[0], "S1 ASC VV (dB)", "gray"),
    (s1[1], "S1 ASC VH (dB)", "gray"),
    (agb, "AGB (Mg/ha)", "viridis"),
]
for a, (img, title, cmap) in zip(ax.flat, panels):
    im = a.imshow(img, cmap=cmap)
    a.set_title(title); a.axis("off")
    if cmap: plt.colorbar(im, ax=a, fraction=0.046)
plt.tight_layout()

In [ ]:
# Target distribution and pixel-level Spearman correlation of each band with AGB (on a sample of tiles)
rng = np.random.default_rng(0)
rows = []
for f in rng.choice(tr_files, size=min(100, len(tr_files)), replace=False):
    d = np.load(f)
    x = np.concatenate([d["s1"], d["s2"]]).reshape(15, -1)[:, ::50]
    rows.append(np.vstack([x, d["agb"].reshape(1, -1)[:, ::50]]))
pix = pd.DataFrame(np.concatenate(rows, 1).T, columns=S1_BANDS + S2_BANDS + ["AGB"]).dropna()

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(pix["AGB"], bins=100); ax[0].set_xlabel("AGB (Mg/ha)"); ax[0].set_title("Target distribution")
corr = pix.corr(method="spearman")["AGB"].drop("AGB")
corr.plot.barh(ax=ax[1]); ax[1].set_title("Spearman correlation with AGB")
plt.tight_layout()

## 4. Training

Each configuration writes `runs/<name>/{config.json, best.pt, log.json}`. Runs with an existing `best.pt` are skipped.

In [ ]:
EXPERIMENTS = {
    "s1_unet":    dict(modality="s1"),
    "s2_unet":    dict(modality="s2"),
    "s1s2_early": dict(modality="s1s2", fusion="early"),
    "s1s2_mid":   dict(modality="s1s2", fusion="mid"),
}

def run_all(experiments, force=False):
    for name, kw in experiments.items():
        if (RUNS / name / "best.pt").exists() and not force:
            print(f"skip {name} (already trained)")
            continue
        train_model(make_cfg(DATA, name=name, runs_dir=RUNS, **{**TRAIN_KW, **kw}))

run_all(EXPERIMENTS)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for name in EXPERIMENTS:
    log = pd.DataFrame(json.loads((RUNS / name / "log.json").read_text()))
    ax[0].plot(log.epoch, log.train_loss, label=name)
    ax[1].plot(log.epoch, log.val_rmse, label=name)
ax[0].set(title="train loss", xlabel="epoch"); ax[1].set(title="val RMSE (Mg/ha)", xlabel="epoch")
ax[1].legend(); plt.tight_layout()

## 5. Test-set evaluation

In [ ]:
results = pd.DataFrame({name: evaluate_run(RUNS / name) for name in EXPERIMENTS}).T
results[["rmse", "mae", "r2", "bias"]].round(3)

In [ ]:
# RMSE per AGB range: shows saturation / underestimation of dense stands
bin_cols = [c for c in results.columns if c.startswith("rmse_")]
results[bin_cols].T.plot.bar(figsize=(9, 4), ylabel="RMSE (Mg/ha)", xlabel="reference AGB bin (Mg/ha)")
plt.tight_layout()

In [ ]:
# Predicted vs reference scatter (subsampled pixels) for each model
from evaluate import load_run, predict_tile
fig, axes = plt.subplots(1, len(EXPERIMENTS), figsize=(4 * len(EXPERIMENTS), 4), sharey=True)
for a, name in zip(axes, EXPERIMENTS):
    _, ds, model, device = load_run(RUNS / name)
    ys, ps = [], []
    for i in range(min(30, len(ds))):
        y, p = predict_tile(model, ds, i, device)
        ys.append(y.flatten()[::20]); ps.append(p.flatten()[::20])
    y, p = torch.cat(ys), torch.cat(ps)
    a.hexbin(y, p, gridsize=60, bins="log", cmap="viridis")
    lim = float(y.max()); a.plot([0, lim], [0, lim], "r--", lw=1)
    a.set(title=name, xlabel="reference AGB")
axes[0].set_ylabel("predicted AGB")
plt.tight_layout()

In [ ]:
plot_predictions(RUNS / "s1s2_mid", idx=(0, 1, 2));

## 6. Contribution analysis

**Inference-time perturbation** on the fused model: each band (and each group, in brackets) is either permuted across
tiles or replaced by its mean, and we record the increase in test RMSE. It needs no retraining, but correlated bands can
stand in for each other.

**Retraining ablation**: drop a group and retrain. This measures the information the model cannot recover from the remaining bands.

In [ ]:
ANALYSIS_RUN = RUNS / "s1s2_mid"
imp_perm = compute_importance(ANALYSIS_RUN, method="permute", repeats=1 if DEBUG else 3)
imp_mean = compute_importance(ANALYSIS_RUN, method="mean")
print(f"baseline RMSE: {imp_perm['baseline_rmse']:.2f}")
plot_importance(imp_perm, ANALYSIS_RUN.name); plot_importance(imp_mean, ANALYSIS_RUN.name);

In [ ]:
# Retraining ablation: drop one group at a time from the fused model
ABLATIONS = {f"s1s2_mid_no{g}": dict(modality="s1s2", fusion="mid", drop=[g])
             for g in ["VV", "VH", "ASC", "Visible", "RedEdge", "NIR", "SWIR"]}
run_all(ABLATIONS)

base = results.loc["s1s2_mid", "rmse"]
abl = pd.Series({name.split("_no")[1]: evaluate_run(RUNS / name)["rmse"] - base for name in ABLATIONS})
perm = pd.Series({g: imp_perm["importance"][f"[{g}]"]["delta_rmse"] for g in abl.index})
cmp = pd.DataFrame({"retrain ablation": abl, "permutation": perm}).sort_values("retrain ablation")
cmp.plot.barh(figsize=(8, 4), xlabel="increase in RMSE (Mg/ha)"); plt.tight_layout()
cmp.round(2)

## 7. Discussion

Points to address:

- **Sensor comparison.** S1 vs S2 vs fused: which range of AGB does each one handle well (see the binned RMSE)?
- **Fusion strategy.** Early vs mid fusion: does a separate encoder per sensor help?
- **Physical consistency.** Do NIR, red edge and SWIR outrank visible bands? Does VH outrank VV? Are ASC and DSC redundant? Does CLP stay near zero?
- **Perturbation vs ablation.** Groups that are large under permutation but small under retraining are redundant with other bands.
- **Limitations and failure modes.** Saturation at high AGB, clouds in a single-date S2 image, missing S1 orbits, small dataset (240 training tiles), spatial autocorrelation.